In [ ]:
# @title Imports
import torch
from datasets import load_dataset
from tqdm import tqdm
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
import numpy as np

In [3]:
# @title Global Vars
special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[EOS]","[BOS]"]
vocab_size=32000

In [4]:
import requests
import zipfile
import os
import json

YEARS = [2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017,2018,2019,2020]
BASE_URL = "https://zenodo.org/records/5528490/files/{year}.zip?download=1"
GOOD_SECTIONS = ["section_1", "section_1A", "section_3", "section_7", "section_7A"]

os.makedirs("edgar_data", exist_ok=True)

for year in YEARS:
    zip_path = f"edgar_data/{year}.zip"
    extract_path = f"edgar_data"

    # --- Download ---
    if not os.path.exists(f"edgar_data/{year}"):
        print(f"Downloading {year}...")
        url = BASE_URL.format(year=year)
        os.system(f"wget -O /content/edgar_data/{year}.zip {url}")
        print(f"  Downloaded {year}.zip")
    if not os.path.exists(extract_path+f"/{year}"):
      print(f"Extracting {year}...")
      with zipfile.ZipFile(zip_path, "r") as z:
          z.extractall(extract_path)
      os.remove(zip_path)
print("Done!")

  Downloaded 2008.zip
Extracting 2008...
  Downloaded 2009.zip
Extracting 2009...
  Downloaded 2010.zip
Extracting 2010...
  Downloaded 2011.zip
Extracting 2011...
  Downloaded 2012.zip
Extracting 2012...
  Downloaded 2013.zip
Extracting 2013...
  Downloaded 2014.zip
Extracting 2014...
  Downloaded 2015.zip
Extracting 2015...
  Downloaded 2016.zip
Extracting 2016...
  Downloaded 2017.zip
Extracting 2017...
  Downloaded 2018.zip
Extracting 2018...
  Downloaded 2019.zip
Extracting 2019...
  Downloaded 2020.zip
Extracting 2020...
Done!


In [5]:
with open(f"/content/processed.jsonl",'w') as out:
  for year in tqdm(YEARS):
    for file in os.listdir(f"edgar_data/{year}"):
      with open(f"edgar_data/{year}/{file}",'r') as f:
        data=json.load(f)

        for section in GOOD_SECTIONS:
          if section in data and len(data[section])>100:
            json.dump({"text":data[section]},out)
            out.write('\n')

100%|██████████| 13/13 [09:25<00:00, 43.52s/it]


In [6]:
# @title Tokenizer
def corpus_iterator(file_path):
  with open(file_path,'r') as f:
    for line in f:
      yield json.loads(line)['text']

In [7]:
len_words=[]
for i in tqdm(corpus_iterator('/content/processed.jsonl')):
  len_words.append(len(i.split()))

434237it [05:24, 1338.77it/s]


In [8]:
total_no_of_training_tokens=sum(len_words)
max_seq_len=max(len_words)
avg_no_of_tokens_per_document=total_no_of_training_tokens/len(len_words)
total_no_of_training_tokens,avg_no_of_tokens_per_document,max_seq_len

(1992552354, 4588.628684335973, 138341)

In [9]:
# @title Training Tokenizer
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()

In [10]:
trainer = BpeTrainer(special_tokens=special_tokens, vocab_size=vocab_size)

In [11]:
tokenizer.train_from_iterator(
    corpus_iterator('/content/processed.jsonl'),
    trainer=trainer
)

tokenizer.save("my_tokenizer.json")

In [12]:
# @title Saving tokenizer and processed data
tokenizer.save("my_tokenizer.json")

In [27]:
Batch_size=100
i=0
batch_docs=[]
EOS_ID=[tokenizer.token_to_id("[EOS]")]
with open("/content/pretraining_dataset.bin",'wb') as f:
  for doc in tqdm(corpus_iterator('/content/processed.jsonl'),total=len(len_words)):
    if doc.strip()=="" :
      continue
    batch_docs.append(doc+" [EOS]")
    i+=1
    if i==Batch_size:
      ids=tokenizer.encode_batch(batch_docs)
      batch_ids=[]
      for id in ids:
        batch_ids.extend(id.ids+EOS_ID)
      arr=np.array(batch_ids,dtype=np.uint16)
      arr.tofile(f)
      batch_docs=[]
      i=0

100%|██████████| 434237/434237 [2:25:55<00:00, 49.59it/s]


In [28]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [38]:
data = np.memmap("/content/pretraining_dataset.bin", dtype=np.uint16, mode='r')

In [44]:
tokenizer.encode(next(iter(corpus_iterator('/content/processed.jsonl'))))

Encoding(num_tokens=2958, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])

In [45]:
[i for i in tokenizer.decode(data[0:10000],skip_special_tokens=False).split() if i=='[EOS]']

['[EOS]', '[EOS]', '[EOS]', '[EOS]']

In [46]:
!mkdir /content/drive/MyDrive/MyLLM2
!cp /content/my_tokenizer.json /content/drive/MyDrive/MyLLM2/
!cp /content/pretraining_dataset.bin /content/drive/MyDrive/MyLLM2/
!cp /content/processed.jsonl /content/drive/MyDrive/MyLLM2/

mkdir: cannot create directory ‘/content/drive/MyDrive/MyLLM2’: File exists


In [ ]:
!ls /content/drive/MyDrive/MyLLM2/

In [48]:
os.path.getsize('/content/drive/MyDrive/MyLLM2/')

4096

In [50]:
os.path.getsize('/content/drive/MyDrive/MyLLM2/my_tokenizer.json')/(1024*1024*1024),os.path.getsize('/content/drive/MyDrive/MyLLM2/pretraining_dataset.bin')/(1024*1024*1024),os.path.getsize('/content/drive/MyDrive/MyLLM2/processed.jsonl')/(1024*1024*1024),

(0.0020073922351002693, 4.404994662851095, 12.282053112983704)